# putEMG — Sequence Model Comparison (Format 3)

Trains and evaluates three sequence model families on temporal EMG feature sequences:

| # | Model | Notes |
|---|-------|-------|
| 1 | **LSTM** | 2-layer bidirectional LSTM, last hidden state → classifier |
| 2 | **GRU** | 2-layer bidirectional GRU, lighter alternative to LSTM |
| 3 | **Transformer** | Encoder-only, mean-pooled output → classifier |

**Input**: `(batch, 26, 192)` — 26 timesteps × 192 features (8 feats × 24 channels) per gesture rep  
**Data split**: 80% train+val · 20% test (train further split 90/10 for early stopping)  
**Prerequisites**: Run `feature_extraction.batch_extract_features(mode='sequence')` first.

In [15]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

from handler import (
    train_loader, val_loader, test_loader,
    SEQ_LEN, INPUT_DIM,
    GESTURE_NAMES, N_CLASSES,
)

In [16]:
# ── Training ───────────────────────────────────────────────────────────────────
MAX_EPOCHS = 50
PATIENCE   = 10     # epochs without improvement before early stopping
MIN_DELTA  = 0.002  # minimum val acc improvement to reset patience

# ── Hyperparameters ────────────────────────────────────────────────────────────
HIDDEN_SIZE  = 128   # LSTM / GRU hidden units
NUM_LAYERS   = 2     # LSTM / GRU stacked layers
D_MODEL      = 128   # Transformer internal dimension
NHEAD        = 4     # Transformer attention heads
TF_LAYERS    = 2     # Transformer encoder layers
DROPOUT      = 0.3
LR           = 1e-3

# ── Weights output ─────────────────────────────────────────────────────────────
WEIGHTS_DIR = 'weights'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {device}')
print(f'SEQ_LEN      : {SEQ_LEN}')
print(f'INPUT_DIM    : {INPUT_DIM}')

Using device : cpu
SEQ_LEN      : 26
INPUT_DIM    : 192


---
## Model Definitions

In [17]:
class LSTMClassifier(nn.Module):
    """
    Bidirectional LSTM.
    Reads the full 26-step sequence and uses the final hidden state
    (both directions concatenated) as the representation for classification.
    """
    def __init__(self, input_dim=INPUT_DIM, hidden_size=HIDDEN_SIZE,
                 num_layers=NUM_LAYERS, dropout=DROPOUT, n_classes=N_CLASSES):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            bidirectional = True,
            dropout = dropout if num_layers > 1 else 0.0,
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, n_classes)   # ×2 for bidirectional

    def forward(self, x):           # x: (batch, seq_len, input_dim)
        _, (h, _) = self.lstm(x)   # h: (num_layers*2, batch, hidden_size)
        # Concatenate the last layer's forward and backward hidden states
        h_fwd = h[-2]              # (batch, hidden_size)
        h_bwd = h[-1]              # (batch, hidden_size)
        out   = torch.cat([h_fwd, h_bwd], dim=1)  # (batch, hidden_size*2)
        return self.classifier(self.dropout(out))

In [18]:
class GRUClassifier(nn.Module):
    """
    Bidirectional GRU.
    Lighter alternative to LSTM — no cell state, fewer parameters.
    Same interface and output shape as LSTMClassifier.
    """
    def __init__(self, input_dim=INPUT_DIM, hidden_size=HIDDEN_SIZE,
                 num_layers=NUM_LAYERS, dropout=DROPOUT, n_classes=N_CLASSES):
        super().__init__()
        self.gru = nn.GRU(
            input_size    = input_dim,
            hidden_size   = hidden_size,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if num_layers > 1 else 0.0,
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, n_classes)

    def forward(self, x):        # x: (batch, seq_len, input_dim)
        _, h = self.gru(x)       # h: (num_layers*2, batch, hidden_size)
        h_fwd = h[-2]
        h_bwd = h[-1]
        out   = torch.cat([h_fwd, h_bwd], dim=1)
        return self.classifier(self.dropout(out))

In [19]:
class SequenceTransformer(nn.Module):
    """
    Encoder-only Transformer.
    Projects input to d_model, runs self-attention over the 26 timesteps,
    then mean-pools the encoder output to get a fixed-size representation.
    """
    def __init__(self, input_dim=INPUT_DIM, d_model=D_MODEL, nhead=NHEAD,
                 num_layers=TF_LAYERS, dropout=DROPOUT, n_classes=N_CLASSES):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer   = nn.TransformerEncoderLayer(
            d_model     = d_model,
            nhead       = nhead,
            dim_feedforward = d_model * 4,
            dropout     = dropout,
            batch_first = True,
        )
        self.encoder    = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x):               # x: (batch, seq_len, input_dim)
        x = self.input_proj(x)          # (batch, seq_len, d_model)
        x = self.encoder(x)             # (batch, seq_len, d_model)
        x = x.mean(dim=1)               # mean pool over sequence → (batch, d_model)
        return self.classifier(x)

---
## Training & Evaluation Helpers

In [20]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_b, y_b in loader:
            preds    = model(X_b.to(device)).argmax(dim=1).cpu()
            correct += (preds == y_b).sum().item()
            total   += y_b.size(0)
    return correct / total


def evaluate_final(model, loader, device, name):
    """Accuracy + confusion matrix on the test set."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            preds = model(X_b.to(device)).argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(y_b)
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
    acc = accuracy_score(all_labels, all_preds)
    print(f'  Test accuracy: {acc * 100:.2f}%')

    cm  = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(cm, display_labels=GESTURE_NAMES).plot(ax=ax, colorbar=False)
    ax.set_title(f'{name} — Test Confusion Matrix')
    plt.tight_layout()
    plt.show()
    return float(acc)


def run_training(model, name):
    """Full training loop with early stopping and best-state checkpointing."""
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = float('-inf')
    best_state   = None
    bad_epochs   = 0

    for epoch in range(MAX_EPOCHS):
        tr_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_acc = evaluate(model, val_loader, device)
        curr_lr = optimizer.param_groups[0]['lr']

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step(val_acc)

        if val_acc >= (best_val_acc - MIN_DELTA):
            bad_epochs = 0
        else:
            bad_epochs += 1

        print(f'  Epoch {epoch+1:3d}: loss={tr_loss:.4f}  val={val_acc*100:.2f}%  '
              f'best={best_val_acc*100:.2f}%  lr={curr_lr:.2e}')

        if bad_epochs >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}.')
            break

    model.load_state_dict(best_state)
    return best_val_acc

---
## 1 — LSTM

Bidirectional 2-layer LSTM. Final hidden states from both directions are
concatenated and passed through a dropout + linear classifier.

In [21]:
print(f"{'='*60}\n  Training: LSTM\n{'='*60}")
lstm = LSTMClassifier().to(device)
run_training(lstm, 'LSTM')

  Training: LSTM
  Epoch   1: loss=1.8286  val=23.94%  best=23.94%  lr=1.00e-03
  Epoch   2: loss=1.8118  val=18.76%  best=23.94%  lr=1.00e-03
  Epoch   3: loss=1.8393  val=21.30%  best=23.94%  lr=1.00e-03
  Epoch   4: loss=1.8697  val=22.21%  best=23.94%  lr=1.00e-03


KeyboardInterrupt: 

In [ ]:
print('--- LSTM — Test Set Evaluation ---')
lstm_acc = evaluate_final(lstm, test_loader, device, 'LSTM')

---
## 2 — GRU

Bidirectional 2-layer GRU. Same structure as the LSTM but without the cell
state — fewer parameters, often comparable accuracy.

In [9]:
print(f"{'='*60}\n  Training: GRU\n{'='*60}")
gru = GRUClassifier().to(device)
run_training(gru, 'GRU')

  Training: GRU
  Epoch   1: loss=1.8900  val=22.11%  best=22.11%  lr=1.00e-03
  Epoch   2: loss=1.7878  val=26.06%  best=26.06%  lr=1.00e-03
  Epoch   3: loss=1.7739  val=21.91%  best=26.06%  lr=1.00e-03
  Epoch   4: loss=1.8172  val=27.48%  best=27.48%  lr=1.00e-03
  Epoch   5: loss=1.7218  val=28.40%  best=28.40%  lr=1.00e-03
  Epoch   6: loss=1.6989  val=26.98%  best=28.40%  lr=1.00e-03
  Epoch   7: loss=1.6913  val=33.98%  best=33.98%  lr=1.00e-03
  Epoch   8: loss=1.6579  val=30.12%  best=33.98%  lr=1.00e-03
  Epoch   9: loss=1.7705  val=23.94%  best=33.98%  lr=1.00e-03


KeyboardInterrupt: 

In [ ]:
print('--- GRU — Test Set Evaluation ---')
gru_acc = evaluate_final(gru, test_loader, device, 'GRU')

---
## 3 — Transformer (Encoder-only)

Projects the 192-dim feature vector at each timestep to `d_model=128`,
runs self-attention over all 26 timesteps, then mean-pools the encoder
output to produce a single representation for classification.

In [ ]:
print(f"{'='*60}\n  Training: Transformer\n{'='*60}")
transformer = SequenceTransformer().to(device)
run_training(transformer, 'Transformer')

In [ ]:
print('--- Transformer — Test Set Evaluation ---')
transformer_acc = evaluate_final(transformer, test_loader, device, 'Transformer')

---
## Results Summary

In [ ]:
results = {
    'LSTM':        lstm_acc,
    'GRU':         gru_acc,
    'Transformer': transformer_acc,
}

print(f"{'Model':<14} {'Test Acc':>10}")
print('-' * 26)
for name, acc in results.items():
    print(f"{name:<14} {acc * 100:>9.2f}%")

fig, ax = plt.subplots(figsize=(7, 4))
names = list(results.keys())
accs  = [v * 100 for v in results.values()]
bars  = ax.bar(names, accs, color=['steelblue', 'darkorange', 'seagreen'])
ax.set_ylim(0, 105)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Sequence Model Comparison — Format 3')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

---
## Save Best Model Weights

In [ ]:
import os
os.makedirs(WEIGHTS_DIR, exist_ok=True)

model_objects = {'LSTM': lstm, 'GRU': gru, 'Transformer': transformer}
best_name     = max(results, key=results.get)
best_acc      = results[best_name]

print(f'Best model : {best_name}')
print(f'Test acc   : {best_acc * 100:.2f}%')

save_path = os.path.join(WEIGHTS_DIR, f'{best_name}_best.pt')
torch.save({
    'model_name': best_name,
    'test_acc':   best_acc,
    'dropout':    DROPOUT,
    'seq_len':    SEQ_LEN,
    'input_dim':  INPUT_DIM,
    'state_dict': model_objects[best_name].state_dict(),
}, save_path)

print(f'Saved → {save_path}')